<style>
/* FABRIC notebook  adaptive theme */
.fab-info    { background-color: #f0f7fb; border-left: 4px solid #1f6a8c; padding: 12px 15px; margin: 15px 0; border-radius: 4px; }
.fab-success { background-color: #e8f5e9; border-left: 4px solid #008e7a; padding: 12px 15px; margin: 15px 0; border-radius: 4px; }
.fab-warning { background-color: #fff8e1; border-left: 4px solid #ff8542; padding: 12px 15px; margin: 15px 0; border-radius: 4px; }
.fab-danger  { background-color: #fce4ec; border-left: 4px solid #b00020; padding: 12px 15px; margin: 15px 0; border-radius: 4px; }
.fab-footer  { background-color: #374955; color: white; padding: 15px 20px; margin: 20px 0; border-radius: 4px; text-align: center; }

@media (prefers-color-scheme: dark) {
  .fab-info    { background-color: #1a2a35; border-color: #5798bc; color: #d0e0eb; }
  .fab-success { background-color: #1a2e25; border-color: #00b89a; color: #c0e0d5; }
  .fab-warning { background-color: #2e2518; border-color: #ff9a5c; color: #e0d0b8; }
  .fab-danger  { background-color: #2e1a1e; border-color: #e53950; color: #e0c0c8; }
  .fab-footer  { background-color: #2a3a45; color: #b0c4d0; }
}
</style>

# Create a Local Ethernet (Layer 2) Network: Manual Configuration

<picture>
  <source srcset="../../images/fabric_logo_light.png" media="(prefers-color-scheme: dark)">
  <img src="../../images/fabric_logo.png" width="300" style="margin-bottom:10px;"/>
</picture>

<div class="fab-info">

**What this notebook does:** This notebook shows how to create an isolated **local Layer 2 Ethernet** network connecting two nodes on the **same** FABRIC site, then **manually** assign IP addresses after the slice becomes active. Manual configuration is ideal when you need specific IP assignments, want to use non-IP protocols, or need to understand the underlying networking primitives.

</div>

## Learning Objectives

<div class="fab-success">

After completing this notebook you will be able to:

1. Create a local L2 network with `add_l2network()` by passing interfaces directly
2. Choose your own IP subnet for the network
3. Manually assign IP addresses to interfaces using `iface.ip_addr_add()`
4. Verify configuration using `ip addr show` and test connectivity with `ping`

</div>

## Prerequisites

<div class="fab-warning">

Before running this notebook you **must**:

1. Complete the [Configure Environment](../../../configure_and_validate/configure_and_validate.ipynb) notebook
2. Be comfortable creating basic slices (see [Hello, FABRIC](../../hello_fabric/hello_fabric.ipynb))

**Tip -- Auto vs. Manual:**
- **Auto** ([auto notebook](./create_l2network_basic_auto.ipynb)): You specify a subnet, FABlib assigns IPs automatically
- **Manual** (this notebook): You assign IPs yourself after the slice is active -- full control

</div>

## Background: Manual L2 Network Configuration

With manual configuration, you submit the slice with **no subnet or IP configuration**. FABRIC creates the L2 Ethernet switch and connects your nodes' NICs to it. After the slice is active, you:

1. **Choose a subnet** (any private range you like, e.g., `192.168.1.0/24`)
2. **Assign IPs** to each interface manually
3. **(Optionally)** deploy non-IP protocols -- the L2 network carries raw Ethernet frames


### When to Use Manual Configuration
- You need **specific IP addresses** (e.g., for reproducible experiments)
- You want to deploy **non-IP protocols** over the L2 link
- You want to understand what happens under the hood

### NIC Component Models

| Model | Speed | Type | Ports |
|-------|-------|------|-------|
| `NIC_Basic` | 100 Gbps | Mellanox ConnectX-6 SR-IOV VF | 1 |
| `NIC_ConnectX_5` | 25 Gbps | Dedicated Mellanox ConnectX-5 | 2 |
| `NIC_ConnectX_6` | 100 Gbps | Dedicated Mellanox ConnectX-6 | 2 |

## What We're Building

In this notebook we will create two nodes on the same site connected by a local L2 Ethernet bridge.

<img src="./figs/slice_topology.png" width="50%">


---

## Step 1: Import FABlib and Verify Configuration

In [ ]:
# Import the FABlib library
from fabrictestbed_extensions.fablib.fablib import FablibManager as fablib_manager

# Create a FABlib manager instance
fablib = fablib_manager()

# Display current configuration (tokens, keys, project info)
fablib.show_config();

## Step 2: Define Slice Parameters

For a local L2 network, both nodes must be on the **same site**.

In [ ]:
# Name for the slice
slice_name = 'MySlice'

# Pick a single random site (both nodes will be here)
site = fablib.get_random_site()
print(f"Site: {site}")

# Node, network, and NIC names
node1_name = 'Node1'
node2_name = 'Node2'
network_name='net1'
node1_nic_name = 'nic1'
node2_nic_name = 'nic2'

## Step 3: Create and Submit the Slice

With manual configuration, we create nodes with NICs and connect them to an L2 network **without** specifying a subnet or setting interface modes. FABRIC just creates the Ethernet switch -- we configure IPs afterward.

In [ ]:
# Create a new empty slice
slice = fablib.new_slice(name=slice_name)

# --- Node1 with a NIC ---
node1 = slice.add_node(name=node1_name, site=site)
# Add a NIC_Basic component and get its first (only) interface
iface1 = node1.add_component(model='NIC_Basic', name=node1_nic_name).get_interfaces()[0]

# --- Node2 with a NIC ---
node2 = slice.add_node(name=node2_name, site=site)
iface2 = node2.add_component(model='NIC_Basic', name=node2_nic_name).get_interfaces()[0]

# --- Create the L2 network, passing both interfaces ---
# Since both interfaces are on the same site, this creates a local Ethernet
# No subnet is specified -- we will configure IPs manually
net1 = slice.add_l2network(name=network_name, interfaces=[iface1, iface2])

# Submit the slice request -- blocks until ready (~3-5 min)
slice.submit();

## Step 4: Manually Configure IP Addresses

Now that the slice is active, the L2 Ethernet is up but the interfaces have **no IP addresses**. Some experiments use L2 networks for non-IP protocols -- if that describes your experiment, you can skip this step.

For most users, the next steps are:
1. Pick a subnet
2. Assign IPs to each interface

### Step 4a: Pick a Subnet

Create a subnet and a list of available IP addresses. All objects are standard Python `ipaddress` objects. You can use either IPv4 or IPv6.

In [ ]:
# Import IP address management library
from ipaddress import ip_address, IPv4Address, IPv6Address, IPv4Network, IPv6Network

# Define a private subnet for our L2 network
subnet = IPv4Network("192.168.1.0/24")

# Generate a list of usable IPs (skip .0 which is the network address)
available_ips = list(subnet)[1:]

### Step 4b: Configure Node1

Get the interface connected to our network, pop an IP from the available pool, and assign it.

In [ ]:
# Get Node1 and its interface on the L2 network
node1 = slice.get_node(name=node1_name)        
node1_iface = node1.get_interface(network_name=network_name) 

# Pop the first available IP (192.168.1.1)
node1_addr = available_ips.pop(0)

# Assign the IP address to the interface with the subnet mask
node1_iface.ip_addr_add(addr=node1_addr, subnet=subnet)

# Verify: show the interface configuration inside the VM
stdout, stderr = node1.execute(f'ip addr show {node1_iface.get_device_name()}')

### Step 4c: Configure Node2

Repeat the same steps for the second node.

In [ ]:
# Get Node2 and its interface on the L2 network
node2 = slice.get_node(name=node2_name)        
node2_iface = node2.get_interface(network_name=network_name)  

# Pop the next available IP (192.168.1.2)
node2_addr = available_ips.pop(0)

# Assign the IP address to the interface
node2_iface.ip_addr_add(addr=node2_addr, subnet=subnet)

# Verify: show the interface configuration
stdout, stderr = node2.execute(f'ip addr show {node2_iface.get_device_name()}')

## Step 5: Run the Experiment

We verify connectivity by pinging Node2 from Node1 over the local L2 network.

In [ ]:
# Ping Node2 from Node1 over the local Ethernet
node1 = slice.get_node(name=node1_name)        

stdout, stderr = node1.execute(f'ping -c 5 {node2_addr}')

<div class="fab-success">

**Success!** If you see ping replies, your two nodes are communicating over the local Layer 2 Ethernet. Since both nodes are on the same site, you should see very low latency (typically sub-millisecond).

</div>

## Step 6: Delete the Slice

<div class="fab-danger">

**Important:** Always delete your slice when you are done. FABRIC is a shared resource -- leaving slices running unnecessarily prevents other researchers from using those resources.

</div>

In [ ]:
# Delete the slice and release all resources
slice = fablib.get_slice(name=slice_name)
slice.delete()

---

## Troubleshooting

| Problem | Possible Cause | Solution |
|---------|---------------|----------|
| `ip_addr_add()` fails | Interface not found or wrong network name | Verify `network_name` matches the name used in `add_l2network()` |
| `ping` fails between nodes | IPs not assigned or interface down | Run `ip addr show` on both nodes to verify configuration |
| Slice stuck in `Configuring` | Site may be busy or down | Try a different site by re-running `get_random_site()` |
| `No resources available` | Site lacks NIC capacity | Choose a different site or try `NIC_ConnectX_6` |
| IP address conflict | Same IP assigned to both nodes | Ensure you `pop()` different IPs from the available list |
| `submit()` times out | Network issue or high demand | Retry with `slice.submit(wait_timeout=600)` |

## FABlib API Reference

The following FABlib methods were used in this notebook:

| Method | Description | Documentation |
|--------|-------------|---------------|
| `fablib.show_config()` | Display current FABlib configuration | [show_config](https://fabric-fablib.readthedocs.io/en/latest/fablib.html#fabrictestbed_extensions.fablib.fablib.FablibManager.show_config) |
| `fablib.get_random_site()` | Get a random site name | [get_random_site](https://fabric-fablib.readthedocs.io/en/latest/fablib.html#fabrictestbed_extensions.fablib.fablib.FablibManager.get_random_site) |
| `fablib.new_slice(name)` | Create a new empty slice | [new_slice](https://fabric-fablib.readthedocs.io/en/latest/fablib.html#fabrictestbed_extensions.fablib.fablib.FablibManager.new_slice) |
| `fablib.get_slice(name)` | Retrieve an existing slice by name | [get_slice](https://fabric-fablib.readthedocs.io/en/latest/fablib.html#fabrictestbed_extensions.fablib.fablib.FablibManager.get_slice) |
| `slice.add_l2network(name, interfaces)` | Add a Layer 2 network | [add_l2network](https://fabric-fablib.readthedocs.io/en/latest/slice.html#fabrictestbed_extensions.fablib.slice.Slice.add_l2network) |
| `slice.add_node(name, site)` | Add a compute node to the slice | [add_node](https://fabric-fablib.readthedocs.io/en/latest/slice.html#fabrictestbed_extensions.fablib.slice.Slice.add_node) |
| `slice.submit()` | Submit the slice for provisioning | [submit](https://fabric-fablib.readthedocs.io/en/latest/slice.html#fabrictestbed_extensions.fablib.slice.Slice.submit) |
| `slice.delete()` | Delete the slice and release resources | [delete](https://fabric-fablib.readthedocs.io/en/latest/slice.html#fabrictestbed_extensions.fablib.slice.Slice.delete) |
| `node.add_component(model, name)` | Add a NIC or other component | [add_component](https://fabric-fablib.readthedocs.io/en/latest/node.html#fabrictestbed_extensions.fablib.node.Node.add_component) |
| `node.get_interface(network_name)` | Get the interface connected to a network | [get_interface](https://fabric-fablib.readthedocs.io/en/latest/node.html#fabrictestbed_extensions.fablib.node.Node.get_interface) |
| `node.execute(command)` | Execute a shell command on a node | [execute](https://fabric-fablib.readthedocs.io/en/latest/node.html#fabrictestbed_extensions.fablib.node.Node.execute) |
| `iface.ip_addr_add(addr, subnet)` | Assign an IP address to an interface | [ip_addr_add](https://fabric-fablib.readthedocs.io/en/latest/interface.html#fabrictestbed_extensions.fablib.interface.Interface.ip_addr_add) |
| `iface.get_device_name()` | Get the Linux device name of an interface | [get_device_name](https://fabric-fablib.readthedocs.io/en/latest/interface.html#fabrictestbed_extensions.fablib.interface.Interface.get_device_name) |

## What's Next?

| Topic | Notebook | What You'll Learn |
|-------|----------|-------------------|
| **L2 Basic Auto** | [create_l2network_basic_auto](./create_l2network_basic_auto.ipynb) | Let FABlib assign IPs automatically |
| **L2 Wide-Area Manual** | [create_l2network_wide_area_manual](../create_l2network_wide_area/create_l2network_wide_area_manual.ipynb) | Extend L2 Ethernet across two sites with manual IPs |
| **L2 Wide-Area Auto** | [create_l2network_wide_area_auto](../create_l2network_wide_area/create_l2network_wide_area_auto.ipynb) | WAN L2 with automatic configuration |
| **FABnet IPv4** | [create_l3network_fabnet_ipv4_manual](../create_l3network_fabnet_ipv4/create_l3network_fabnet_ipv4_manual.ipynb) | Layer 3 networking with manual config |
| **Hello FABRIC** | [hello_fabric](../../hello_fabric/hello_fabric.ipynb) | Start from the basics |